> **Note**: This notebook requires the outputs of `01_video_processing.ipynb`:
> - `data/all_observations.csv` — full per-fish summary for all videos
> - `data/derived/all_distances.csv` — pairwise distances per video
> - `data/usable_videos.xlsx` — video metadata
> 
> Outputs produced by this notebook:
> - `data/filtered_observations.csv` — observations used in analysis (social responders + non-responders)
> - `data/derived/model_input.csv` — per-fish model input with neighbour influence data
> - `data/derived/response_videos.csv` — list of videos with a response event

# Prepare model input dataset

Overview - 02. Prepare model input dataset

- Filter observations to social responders and non-responders
- For each focal fish, calculate the influence of each responding neighbour
  (distance, response timing, speed) at the moment of their response
- Save model input dataset

## Load packages and paths

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
path = ROOT

## Filter observations

In [ ]:
overall_summaries = pd.read_csv(f'{path}/data/all_observations.csv')

# Keep social responders (SRL, SRNL) and non-responders in line of sight (NRL_r, NRNL_r)
# Exclude fish larger than 500mm (likely mis-tracked)
filtered = overall_summaries[overall_summaries['Response_Category'].isin(['SRL', 'SRNL', 'NRL_r', 'NRNL_r'])]
filtered = filtered[filtered['Size'] < 500]
filtered = filtered.reset_index(drop=True)
filtered.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

filtered.to_csv(f'{path}/data/filtered_observations.csv', index=False)

## Define neighbour influence function

In [ ]:
def calculate_influence_response(df_summary, df_distances, focal_fish):

    response_time = df_summary.loc[df_summary['Fish_ID'] == focal_fish, 'Responseframe_cam3'].values[0]

    summary_cut = df_summary.loc[df_summary['Fish_ID'] != focal_fish]
    summary_cut = summary_cut.loc[summary_cut['Response_Category'].isin(['FR', 'SRL', 'SRNL', 'NRL_r', 'NRNL_r'])]

    columns = ['Dep_ID', 'Video_ID', 'Focal_Fish_ID', 'Focal_Species', 'Focal_Response_Category', 'Focal_Stimulus', 'Focal_Responseframe_cam3', 'Focal_Speed', 'Focal_Distance_Coral', 'Focal_Neighbour_Proximity',
                'Input_Fish_ID', 'Input_Species', 'Input_Response_Category', 'Input_Responseframe_cam3', 'Input_Distance', 'Input_Speed', 'Input_Size', 'Influence']
    input_timeline = pd.DataFrame(columns=columns)

    input_timeline['Dep_ID'] = summary_cut['Dep_ID'].copy()
    input_timeline['Video_ID'] = summary_cut['Video_ID'].copy()
    input_timeline['Focal_Fish_ID'] = [focal_fish] * len(summary_cut)
    input_timeline['Focal_Species'] = df_summary.loc[df_summary['Fish_ID'] == focal_fish, 'Species'].values[0] 
    input_timeline['Focal_Response_Category'] = df_summary.loc[df_summary['Fish_ID'] == focal_fish, 'Response_Category'].values[0]
    input_timeline['Focal_Stimulus'] = 'Sees_Stimulus' if df_summary.loc[df_summary['Fish_ID'] == focal_fish, 'Response_Category'].values[0] in ['SRL', 'NRL_r'] else 'No_Stimulus'
    input_timeline['Focal_Responseframe_cam3'] = response_time
    input_timeline['Focal_Speed'] = df_summary.loc[df_summary['Fish_ID'] == focal_fish, 'Speed'].values[0]
    input_timeline['Focal_Distance_Coral'] = df_summary.loc[df_summary['Fish_ID'] == focal_fish, 'Distance_to_Coral'].values[0]
    input_timeline['Focal_Neighbour_Proximity'] = df_summary.loc[df_summary['Fish_ID'] == focal_fish, 'Neighbour_Proximity'].values[0]
    input_timeline['Input_Fish_ID'] = summary_cut['Fish_ID'].copy()
    input_timeline['Input_Species'] = summary_cut['Species'].copy()
    input_timeline['Input_Response_Category'] = summary_cut['Response_Category'].copy()
    input_timeline['Input_Responseframe_cam3'] = summary_cut['Responseframe_cam3'].copy()
    input_timeline['Input_Speed'] = summary_cut['Speed'].copy()
    input_timeline['Input_Size'] = summary_cut['Size'].copy()
    
    input_distance = []

    distance_row = df_distances.loc[df_distances['Fish_ID'] == focal_fish]
    for index, row in summary_cut.iterrows():
        input_distance.append(distance_row[f'Distance_to_{row["Fish_ID"]}'].values[0])
    
    input_timeline['Input_Distance'] = input_distance

    input_timeline['Influence'] = 0 

    if df_summary.loc[df_summary['Fish_ID'] == focal_fish, 'Response_Category'].values[0] in ['SRL', 'SRNL']:
        condition = (input_timeline['Input_Responseframe_cam3'] <= response_time - sensory_motor_delay) & \
            (input_timeline['Input_Response_Category'].isin(['FR', 'SRL', 'SRNL']))
    elif df_summary.loc[df_summary['Fish_ID'] == focal_fish, 'Response_Category'].values[0] in ['NRL_r', 'NRNL_r']:
        condition = (input_timeline['Input_Response_Category'].isin(['FR', 'SRL', 'SRNL']))

    input_timeline.loc[condition, 'Influence'] = 1

    return input_timeline

## Calculate model input

In [ ]:
usable_videos = pd.read_excel(f'{path}/data/usable_videos.xlsx')
overall_summaries = pd.read_csv(f'{path}/data/all_observations.csv')
all_distances = pd.read_csv(f'{path}/data/derived/all_distances.csv')

response_videos = usable_videos.loc[usable_videos['Response'] == 'Yes']

In [ ]:
input_data_list = []

for index, row in response_videos.iterrows():
    Overall_Summary = overall_summaries.loc[overall_summaries['Video_ID'] == row['Video_ID']]
    Overall_Distance = all_distances.loc[all_distances['Video_ID'] == row['Video_ID']]
    
    for index_fish, row_fish in Overall_Summary.iterrows():
        if row_fish['Response_Category'] in ['SRL', 'SRNL', 'NRL_r', 'NRNL_r']:
            input_data_list.append(calculate_influence_response(Overall_Summary, Overall_Distance, row_fish['Fish_ID']))

# Concatenate all the dataframes in the list into a single dataframe
input_data = pd.concat(input_data_list, ignore_index=True)

# Export to Excel
input_data.to_csv(f'{path}/data/derived/model_input.csv', index=False)

## Save response video list

In [ ]:
response_videos.to_csv(f'{path}/data/derived/response_videos.csv', index=False)